# Global metrics and invariant measures over long forecast horizons

In [2]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import sys

parrot_color = np.array([46, 69, 81]) / 255
dynamix_color = np.array([152, 199, 71]) / 255
simplex_color = np.array([234, 57, 89]) / 255
chronos_color = np.array([80, 155, 143]) / 255

from scipy.stats import spearmanr, pearsonr

def corr_se(r, n):
    """
    Args:
        r (float): Pearson correlation coefficient.
        n (int): Sample size.

    Returns:
        float: Approximate standard error of r.
    """
    return (1 - r**2) / np.sqrt(n - 3)


plt.rcParams["font.family"] = "Helvetica"

First, we check that the internal Lyapunov estimators are behaving as expected.

In [31]:
from dysts.systems import get_attractor_list
import dysts.flows as dfl
from scipy.stats import spearmanr, pearsonr

from dysts.analysis import max_lyapunov_exponent_rosenstein, max_lyapunov_exponent_rosenstein_multivariate

eq_names = get_attractor_list()
all_lyap_true = list()
all_lyap1 = list()
all_lyap2 = list()
for eq_name in eq_names:
    print(eq_name, flush=True)
    eq = getattr(dfl, eq_name)()
    all_lyap_true.append(eq.maximum_lyapunov_estimated)

    tpts, sol = eq.make_trajectory(1000, standardize=True, return_times=True)
    dt = tpts[1] - tpts[0]
    all_lyap1.append(max_lyapunov_exponent_rosenstein(sol) / dt)
    all_lyap2.append(max_lyapunov_exponent_rosenstein_multivariate(sol) / dt)

    if len(all_lyap_true) > 3:
        print(pearsonr(all_lyap_true, all_lyap1)[0], flush=True)
        print(pearsonr(all_lyap_true, all_lyap2)[0], flush=True)




Aizawa
AnishchenkoAstakhov
Arneodo
ArnoldBeltramiChildress
0.8868028636106682
0.9733888320064552
ArnoldWeb
0.8685352212608615
0.9798591966656846
AtmosphericRegime
0.8380471961164341
0.9634734139307791
BeerRNN
0.8395948728930074
0.9669528612955002
BelousovZhabotinsky
0.9996171207295546
0.9999323108328098
BickleyJet
0.9996142516381462
0.9999290730847522
Blasius
0.9996168671260536
0.9999296467861603
BlinkingRotlet
0.9995751615856194
0.9999033663094752
BlinkingVortex
0.9995249222758256
0.99984232304979
Bouali
0.9995284905452715
0.9998360966073986
Bouali2
0.9995250460243787
0.9998367329359478
BurkeShaw
0.9994322199086529
0.9987126075721515
CaTwoPlus
0.9993355178680189
0.9986370046891726
CaTwoPlusQuasiperiodic
0.9987668418216669
0.9971971998558914
CellCycle
0.9987323940073013
0.997175826445514
CellularNeuralNetwork
0.9986775267060901
0.9971711911602198
Chen
0.9983102578824988
0.9964975583181661
ChenLee
0.9982921115024029
0.9965006568097581
Chua
0.9982920435019643
0.9964946843035402
Circadian

## Long term metrics

Check that the Lyapunov exponents of long forecasts are close to the true Lyapunov exponents.

In [3]:
import sys
# sys.path.append('.')
sys.path.append('../DynaMix/')

import torch
from src.model.forecaster import DynaMixForecaster
from src.utilities.utilities import load_hf_model

# Load the pre-trained model
model = load_hf_model("dynamix-3d-alrnn-v1.0")
model.eval() # Set model to evaluation mode
forecaster = DynaMixForecaster(model) # Initialize the forecaster

/Users/william/program_repos/parroting/benchmark/private_development/../DynaMix/src/model/dynamix.py:95: RuntimeWarning: divide by zero encountered in matmul
  K = R.T @ R / M + np.eye(M)
/Users/william/program_repos/parroting/benchmark/private_development/../DynaMix/src/model/dynamix.py:95: RuntimeWarning: overflow encountered in matmul
  K = R.T @ R / M + np.eye(M)
/Users/william/program_repos/parroting/benchmark/private_development/../DynaMix/src/model/dynamix.py:95: RuntimeWarning: invalid value encountered in matmul
  K = R.T @ R / M + np.eye(M)


In [4]:
import sys
sys.path.append('../..')
from models.parrot import context_parroting_forecast

In [ ]:
from dysts.systems import get_attractor_list
import dysts.flows as dfl
from scipy.stats import spearmanr, pearsonr

from dysts.analysis import max_lyapunov_exponent_rosenstein_multivariate, gp_dim
from dysts.metrics import average_hellinger_distance, estimate_kl_divergence

CONTEXT_LENGTH = 2000
FORECAST_LENGTH = 10000

eq_names = get_attractor_list()
# all_lyap_true, all_lyap_parrot, all_lyap_dynamix = list(), list(), list()
all_lyap_true, all_lyap_parrot, all_lyap_dynamix = dict(), dict(), dict()
all_cdim_true, all_cdim_parrot, all_cdim_dynamix = dict(), dict(), dict()
all_hellinger_dynamix, all_hellinger_parrot = list(), list()
all_kl_dynamix, all_kl_parrot = list(), list()
for eq_name in eq_names:

    if eq_name == "SprottG":
        continue

    print(eq_name, flush=True)
    eq = getattr(dfl, eq_name)()

    try:
        ## Make a long trajectory and split it into context and true trajectory
        np.random.seed(0)
        eq.ic += np.random.randn(len(eq.ic)) * 0.3
        traj = eq.make_trajectory(500 + CONTEXT_LENGTH + FORECAST_LENGTH, standardize=True)[500:]
        traj_context = traj[:CONTEXT_LENGTH, :]
        traj_true = traj[CONTEXT_LENGTH:, :]

        ## Generate predictions using context parroting
        traj_pred = list()
        for mode in range(traj_context.shape[1]):
            traj_pred_mode = context_parroting_forecast(
                traj_context[:, mode], forecast_total_length=FORECAST_LENGTH
            )[2]
            traj_pred.append(traj_pred_mode.copy())
        traj_pred_parrot = np.array(traj_pred).T

        ## Generate predictions using DynaMix
        context_traj_tensor  = torch.tensor(traj_context)
        with torch.no_grad(): 
            reconstruction = forecaster.forecast(
                context=context_traj_tensor,
                horizon=FORECAST_LENGTH, # Match the horizon of the ground truth
                standardize=False,
            )
        traj_pred_dynamix = reconstruction.detach().numpy()

        ## Compute the Lyapunov exponents
        all_lyap_true[eq_name] = max_lyapunov_exponent_rosenstein_multivariate(traj_true)
        all_lyap_dynamix[eq_name] = max_lyapunov_exponent_rosenstein_multivariate(traj_pred_dynamix)
        all_lyap_parrot[eq_name] = max_lyapunov_exponent_rosenstein_multivariate(traj_pred_parrot)

        ## Compute the GP dimension
        all_cdim_true[eq_name] = gp_dim(traj_true)
        all_cdim_dynamix[eq_name] = gp_dim(traj_pred_dynamix)
        all_cdim_parrot[eq_name] = gp_dim(traj_pred_parrot)

        # ## Compute the Hellinger distance and KL divergence
        all_hellinger_dynamix.append(average_hellinger_distance(traj_true, traj_pred_dynamix)) 
        all_hellinger_parrot.append(average_hellinger_distance(traj_true, traj_pred_parrot)) 
        all_kl_dynamix.append(estimate_kl_divergence(traj_true, traj_pred_dynamix, sigma_scale=None))
        all_kl_parrot.append(estimate_kl_divergence(traj_true, traj_pred_parrot, sigma_scale=None))
    except:
        print(f"Error for {eq_name}")


all_hellinger_dynamix = np.array(all_hellinger_dynamix)
all_hellinger_parrot = np.array(all_hellinger_parrot)
all_kl_dynamix = np.array(all_kl_dynamix)
all_kl_parrot = np.array(all_kl_parrot)


np.array(all_hellinger_dynamix).dump('all_hellinger_dynamix2.pkl')
np.array(all_hellinger_parrot).dump('all_hellinger_parrot2.pkl')
np.array(all_kl_dynamix).dump('all_kl_dynamix2.pkl')
np.array(all_kl_parrot).dump('all_kl_parrot2.pkl')

# Save the results
import pandas as pd
df_all = pd.concat([
    pd.DataFrame.from_dict(all_lyap_true, orient='index'),
    pd.DataFrame.from_dict(all_lyap_dynamix, orient='index'),
    pd.DataFrame.from_dict(all_lyap_parrot, orient='index')
], axis=1, join='inner', ignore_index=True)
## name columns
df_all.columns = ['true', 'dynamix', 'parrot']
## drop columnns with infinite values
df_all = df_all.replace([np.inf, -np.inf], np.nan).dropna()
df_all.to_csv('all_lyap_true_dynamix_parrot2.csv')


# Save the results
import pandas as pd
df_all_cdim = pd.concat([
    pd.DataFrame.from_dict(all_cdim_true, orient='index'),
    pd.DataFrame.from_dict(all_cdim_dynamix, orient='index'),
    pd.DataFrame.from_dict(all_cdim_parrot, orient='index')
], axis=1, join='inner', ignore_index=True)
## name columns
df_all_cdim.columns = ['true', 'dynamix', 'parrot']
# ## drop columnns with infinite values
df_all_cdim = df_all_cdim.replace([np.inf, -np.inf], np.nan).dropna()
df_all_cdim.to_csv('all_cdim_true_dynamix_parrot2.csv')


    

Aizawa
AnishchenkoAstakhov
Arneodo
ArnoldBeltramiChildress
ArnoldWeb
AtmosphericRegime
BeerRNN
BelousovZhabotinsky
BickleyJet
Blasius
BlinkingRotlet
BlinkingVortex
Bouali
Bouali2
BurkeShaw
CaTwoPlus
CaTwoPlusQuasiperiodic
CellCycle
CellularNeuralNetwork
Chen
ChenLee
Chua
CircadianRhythm
CoevolvingPredatorPrey
Colpitts
Coullet
Error for Coullet
Dadras
DequanLi
DoubleGyre
DoublePendulum
Duffing
ExcitableCell
Finance
FluidTrampoline
ForcedBrusselator
ForcedFitzHughNagumo
ForcedVanDerPol
GenesioTesi
GlycolyticOscillation
GuckenheimerHolmes
Hadley
Halvorsen
HastingsPowell
HenonHeiles
Error for HenonHeiles
HindmarshRose
Hopfield
HyperBao
HyperCai
HyperJha
HyperLorenz
HyperLu
HyperPang
HyperQi
HyperRossler
HyperWang
HyperXu
HyperYan
HyperYangChen
IkedaDelay
InteriorSquirmer
IsothermalChemical
ItikBanksTumor
JerkCircuit
KawczynskiStrizhak
Laser
LidDrivenCavityFlow
LiuChen
Lorenz
Lorenz84
Lorenz96
LorenzBounded
LorenzCoupled
LorenzStenflo
LuChen
LuChenCheng
MacArthur
MackeyGlass
MooreSpiegel
Mu

/Users/william/program_repos/parroting/.venv/lib/python3.13/site-packages/scipy/integrate/_ode.py:438: UserWarning: vode: Excess work done on this call. (Perhaps wrong MF.)
  self._y, self.t = mth(self.f, self.jac or (lambda: None),
/Users/william/program_repos/parroting/.venv/lib/python3.13/site-packages/numpy/_core/_methods.py:197: RuntimeWarning: overflow encountered in multiply
  x = um.multiply(x, x, out=x)
/Users/william/program_repos/parroting/.venv/lib/python3.13/site-packages/statsmodels/tsa/stattools.py:702: RuntimeWarning: invalid value encountered in divide
  acf = avf[: nlags + 1] / avf[0]


/Users/william/program_repos/parroting/.venv/lib/python3.13/site-packages/dysts/analysis.py:209: RuntimeWarning: divide by zero encountered in log10
  rvals = np.logspace(np.log10(0.1 * std), np.log10(0.5 * std), nmax)
/Users/william/program_repos/parroting/.venv/lib/python3.13/site-packages/numpy/_core/function_base.py:146: RuntimeWarning: invalid value encountered in subtract
  delta = np.subtract(stop, start, dtype=type(dt))


Error for VossDelay
WangSun
WindmiReduced
YuWang
YuWang2
ZhouChen


In [12]:
import pandas as pd
# df_all_cdim = pd.read_csv('all_cdim_true_dynamix_parrot.csv')
corr_dynamix = pearsonr(df_all_cdim["true"], df_all_cdim["dynamix"])[0]
corr_parrot = pearsonr(df_all_cdim["true"], df_all_cdim["parrot"])[0]
se_dynamix = corr_se(corr_dynamix, len(df_all_cdim))
se_parrot = corr_se(corr_parrot, len(df_all_cdim))
print(f"{corr_dynamix:.3f} ± {se_dynamix:.3f}")
print(f"{corr_parrot:.3f} ± {se_parrot:.3f}")


0.441 ± 0.073
0.832 ± 0.028


In [ ]:
import pandas as pd
df_all = pd.read_csv('all_lyap_true_dynamix_parrot.csv')
corr_dynamix = pearsonr(df_all["true"], df_all["dynamix"])[0]
corr_parrot = pearsonr(df_all["true"], df_all["parrot"])[0]
se_dynamix = corr_se(corr_dynamix, len(df_all))
se_parrot = corr_se(corr_parrot, len(df_all))
print(f"{corr_dynamix:.3f} ± {se_dynamix:.3f}")
print(f"{corr_parrot:.3f} ± {se_parrot:.3f}")


0.278 ± 0.100
0.328 ± 0.097


In [ ]:
all_hellinger_dynamix = np.load('all_hellinger_dynamix.pkl', allow_pickle=True)
all_hellinger_parrot = np.load('all_hellinger_parrot.pkl', allow_pickle=True)
all_kl_dynamix = np.load('all_kl_dynamix.pkl', allow_pickle=True)
all_kl_parrot = np.load('all_kl_parrot.pkl', allow_pickle=True)

print(f"{np.mean(all_hellinger_dynamix):.3f} ± {np.std(all_hellinger_dynamix):.3f}")
print(f"{np.mean(all_hellinger_parrot):.3f} ± {np.std(all_hellinger_parrot):.3f}")
print(f"{np.mean(all_kl_dynamix):.3f} ± {np.std(all_kl_dynamix):.3f}")
print(f"{np.mean(all_kl_parrot):.3f} ± {np.std(all_kl_parrot):.3f}")



0.595 ± 0.166
0.591 ± 0.198
4.251 ± 4.584
4.302 ± 5.983
